# 🌳 Notebook 1: Build a Merkle Tree from Scratch

**The big question:** *"Two replicas have a million keys each. How do I check whether they have the same data, without sending all of it over the network?"*

A **Merkle tree** is a binary tree where every leaf is the hash of one piece of data, and every internal node is the hash of its two children. The root hash is one tiny number (32 bytes for SHA-256) that depends on **every** leaf. Change a single byte anywhere and the root hash changes.

That's the magic: comparing one number tells you whether two large datasets are identical.

In this notebook we:

1. Hash a list of values into leaves.
2. Pair them up and hash again, recursively, until we have one root.
3. Show that one bit-flip in the data flips the root hash.

## Learning objectives
- Implement a Merkle tree using `hashlib`.
- Convince yourself any change → different root.
- Set up the data structure we'll use in notebook 2 to find *which* keys differ.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/merkle-trees
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).

In [ ]:
import hashlib

def H(b: bytes) -> bytes:
    return hashlib.sha256(b).digest()

def merkle(leaves: list[bytes]) -> list[list[bytes]]:
    '''Build a Merkle tree. Returns a list of levels, leaves at index 0, root at the top.'''
    if not leaves:
        return [[H(b"")]]
    level = [H(x) for x in leaves]            # hash each leaf
    levels = [level]
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]       # duplicate last to pad to even
        nxt = [H(level[i] + level[i+1]) for i in range(0, len(level), 2)]
        levels.append(nxt)
        level = nxt
    return levels

def root(leaves):
    return merkle(leaves)[-1][0]

data = [b"alice=100", b"bob=50", b"carol=75", b"dave=20"]
levels = merkle(data)
for i, lvl in enumerate(levels):
    print(f"level {i}: {[h.hex()[:10] for h in lvl]}")
print("\nroot:", root(data).hex())

In [ ]:
# Flip one tiny thing — the root changes completely.
data2 = [b"alice=100", b"bob=51", b"carol=75", b"dave=20"]   # bob lost a dollar
print("root before:", root(data).hex())
print("root after :", root(data2).hex())

## 🪜 Why a tree, not just `hash(everything)`?

A single hash of all the data also detects differences. But a tree gives you something extra: when two roots differ, you can **walk down** to find *which leaves* differ in `O(log N)` steps instead of `O(N)`. We'll see that next.